#  Order Payments - Bronze Ingestion

## Imports


In [0]:
import uuid
from pyspark.sql.functions import col, current_timestamp, lit
from pyspark.sql.types import StringType, IntegerType, DecimalType, StructField, StructType

## Configuration

In [0]:
environment = "dev"

catalog = f"ecommerce_{environment}"
schema = "bronze"
table_name = "olist_order_payments"
target_table = f"{catalog}.{schema}.{table_name}"

source_system = "olist"
source_dataset = "order_payments"

source_path = (
    f"/Volumes/{catalog}/landing/raw_files/"
    f"{source_system}/{source_dataset}/"
)

checkpoint_path = (
    f"/Volumes/{catalog}/ops/runtime_state/"
    f"autoloader/{source_system}/{source_dataset}/checkpoint/"
)

schema_location = (
    f"/Volumes/{catalog}/ops/runtime_state/"
    f"autoloader/{source_system}/{source_dataset}/schema/"
)

run_id = str(uuid.uuid4())

## Schema Definition

In [0]:
source_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("payment_sequential", IntegerType(), True),
    StructField("payment_type", StringType(), True),
    StructField("payment_installments", IntegerType(), True),
    StructField("payment_value", DecimalType(8,2), True),

])

## Read with Auto Loader

In [0]:
source_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .option("header", "true")
    .schema(source_schema)
    .load(source_path)
)

## Add Bronze metadata

In [0]:
bronze_df = (
    source_df
    .withColumn("source_file_path", col("_metadata.file_path"))
    .withColumn("source_file_modification_time", col("_metadata.file_modification_time"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("ingestion_run_id", lit(run_id))
    .withColumn("source_system", lit(source_system))
    .withColumn("source_dataset", lit(source_dataset))
)

## Write to the Bronze table

In [0]:
query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

In [0]:
spark.table(target_table).printSchema()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: decimal(8,2) (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(spark.table(target_table).limit(5))

order_id,payment_sequential,payment_type,payment_installments,payment_value,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments
25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments
ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments
42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments


In [0]:
spark.table(target_table).count()

103886